# Local Deepfake Pipeline

This notebook is the local-PC version of the Kaggle pipeline. It samples 2 videos from each leaf folder, then runs the video, audio, fusion, and single-video inference stages on that sampled subset.

## Run order

1. Build the uniform subset.
2. Train the video model.
3. Train the audio model.
4. Train the fusion model.
5. Run single-video inference.

In [1]:
import json
import os
import random
import shutil
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

def resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct_script = candidate / "train_mobilevit_video_deepfake.py"
        if direct_script.exists():
            return candidate
        nested_script = candidate / "FakeBDTeen" / "train_mobilevit_video_deepfake.py"
        if nested_script.exists():
            return candidate / "FakeBDTeen"
    raise RuntimeError("Could not find the FakeBDTeen project root from the current working directory.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = resolve_project_root(Path.cwd())
RAW_DATASET_ROOT = PROJECT_ROOT
WORK_DIR = PROJECT_ROOT / "local_run_outputs"
SAMPLED_DATASET_ROOT = WORK_DIR / "sampled_uniform_480_videos"
SAMPLED_MANIFEST_PATH = WORK_DIR / "sample_manifest.json"

VIDEO_SCRIPT = PROJECT_ROOT / "train_mobilevit_video_deepfake.py"
AUDIO_SCRIPT = PROJECT_ROOT / "train_wav2vec2_audio_deepfake.py"
FUSION_SCRIPT = PROJECT_ROOT / "train_fusion_two_head_multimodal.py"
INFER_SCRIPT = PROJECT_ROOT / "infer_two_head_multimodal.py"

VIDEO_OUT = WORK_DIR / "outputs_video"
AUDIO_OUT = WORK_DIR / "outputs_audio"
FUSION_OUT = WORK_DIR / "outputs_fusion_two_head"

CONFIG = {
    "dataset_root": str(SAMPLED_DATASET_ROOT),
    "working_dir": str(WORK_DIR),
    "num_frames": 8,
    "image_size": 224,
    "clip_seconds": 5,
    "max_train_samples": 1200,
    "max_val_samples": 160,
    "max_test_samples": 160,
    "video_epochs": 10,
    "audio_epochs": 20,
    "fusion_epochs": 35,
    "force_retrain": False,
}

AUDIO_CONFIG = {
    "ffmpeg_timeout": 45,
    "force_cpu": False,
    "max_train_samples": 0,
    "max_val_samples": 0,
    "max_test_samples": 0,
    "force_retrain": False,
}

FUSION_CONFIG = {
    "output_dir": str(FUSION_OUT),
    "epochs": 35,
    "batch_size": 128,
    "lr": 7e-4,
    "weight_decay": 1e-2,
    "num_workers": 2,
    "d_model": 256,
    "num_attn_heads": 8,
    "dropout": 0.2,
    "contrastive_weight": 0.05,
    "grad_clip_norm": 1.0,
    "early_stop_patience": 8,
    "resume": True,
    "force_retrain": False,
}

INFER_CONFIG = {
    "video_path": str(SAMPLED_DATASET_ROOT / "input_video.mp4"),
    "fusion_checkpoint": str(FUSION_OUT / "best_checkpoint.pt"),
    "output_json": str(WORK_DIR / "single_video_prediction.json"),
    "num_frames": CONFIG["num_frames"],
    "image_size": CONFIG["image_size"],
    "clip_seconds": CONFIG["clip_seconds"],
    "audio_duration_sec": 10,
    "audio_sr": 16000,
    "ffmpeg_timeout": AUDIO_CONFIG["ffmpeg_timeout"],
    "force_cpu": AUDIO_CONFIG["force_cpu"],
}

WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Raw dataset root:", RAW_DATASET_ROOT)
print("Sampled dataset root:", SAMPLED_DATASET_ROOT)
print("Video script exists:", VIDEO_SCRIPT.exists())
print("Audio script exists:", AUDIO_SCRIPT.exists())
print("Fusion script exists:", FUSION_SCRIPT.exists())
print("Inference script exists:", INFER_SCRIPT.exists())
print(json.dumps(CONFIG, indent=2))

Project root: f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen
Raw dataset root: f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen
Sampled dataset root: f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\sampled_uniform_480_videos
Video script exists: True
Audio script exists: True
Fusion script exists: True
Inference script exists: True
{
  "dataset_root": "f:\\Sixth Semester\\DEEPfake Papers\\FakeBDTeen\\FakeBDTeen\\local_run_outputs\\sampled_uniform_480_videos",
  "working_dir": "f:\\Sixth Semester\\DEEPfake Papers\\FakeBDTeen\\FakeBDTeen\\local_run_outputs",
  "num_frames": 8,
  "image_size": 224,
  "clip_seconds": 5,
  "max_train_samples": 1200,
  "max_val_samples": 160,
  "max_test_samples": 160,
  "video_epochs": 10,
  "audio_epochs": 20,
  "fusion_epochs": 35,
  "force_retrain": false
}


In [2]:
from collections import Counter

def build_uniform_video_subset(raw_root: Path, per_folder: int, seed: int):
    rng = random.Random(seed)
    folder_to_videos = defaultdict(list)

    for ext in ("*.mp4", "*.MP4"): 
        for video_path in raw_root.rglob(ext):
            folder_to_videos[video_path.parent].append(video_path)

    if not folder_to_videos:
        raise RuntimeError(f"No videos found under {raw_root}")

    selected_files = []
    folder_stats = []
    for folder in sorted(folder_to_videos, key=lambda p: str(p).lower()):
        videos = sorted(folder_to_videos[folder], key=lambda p: p.name.lower())
        if len(videos) < per_folder:
            raise RuntimeError(f"Folder {folder} has only {len(videos)} videos; cannot sample {per_folder}.")
        chosen = sorted(rng.sample(videos, per_folder), key=lambda p: p.name.lower())
        selected_files.extend(chosen)
        folder_stats.append({
            "folder": str(folder.relative_to(raw_root)),
            "available": len(videos),
            "selected": len(chosen),
        })

    return selected_files, folder_stats

def materialize_subset(selected_files, raw_root: Path, sampled_root: Path):
    if sampled_root.exists():
        shutil.rmtree(sampled_root)
    sampled_root.mkdir(parents=True, exist_ok=True)

    for src in selected_files:
        dst = sampled_root / src.relative_to(raw_root)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

selected_files, folder_stats = build_uniform_video_subset(RAW_DATASET_ROOT, 2, SEED)
materialize_subset(selected_files, RAW_DATASET_ROOT, SAMPLED_DATASET_ROOT)
sample_manifest = [
    {
        "relative_path": str(src.relative_to(RAW_DATASET_ROOT)),
        "folder": str(src.parent.relative_to(RAW_DATASET_ROOT)),
    }
    for src in selected_files
]
SAMPLED_MANIFEST_PATH.write_text(json.dumps(sample_manifest, indent=2))

folder_summary = Counter(item['folder'].split('/')[0] for item in folder_stats)
print("Folders discovered:", len(folder_stats))
print("Videos selected:", len(selected_files))
print("Expected videos:", len(folder_stats) * 2)
print("Per-top-level-folder counts:", dict(folder_summary))
print("Sample manifest saved to:", SAMPLED_MANIFEST_PATH)
print("Sampled root contains only the selected files.")

Folders discovered: 240
Videos selected: 480
Expected videos: 480
Per-top-level-folder counts: {'Boys\\Fake_Video + Fake_Audio\\Bangla\\B00': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B01': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B02': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B03': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B04': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B05': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B06': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B07': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B08': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B09': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B10': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B11': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B12': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B13': 1, 'Boys\\Fake_Video + Fake_Audio\\Bangla\\B14': 1, 'Boys\\Fake_Video + Fake_Audio\\English\\B00': 1, 'Boys\\Fake_Video + Fake_Audio\\English\\B01': 1, 'Boys\\Fake_Video + Fake_Audio\\English\\B02': 1, 'Boys\\Fake_Video +

## Local prerequisites

Make sure `ffmpeg` is installed and available on your PATH before running audio extraction. If a package is missing, install it in the notebook environment first.

In [3]:
ffmpeg_path = shutil.which("ffmpeg")
print("ffmpeg found:", ffmpeg_path or "not found")
if ffmpeg_path is None:
    print("Audio training will fail until ffmpeg is installed and added to PATH.")

ffmpeg found: C:\Users\ashab\AppData\Local\Microsoft\WinGet\Links\ffmpeg.EXE


In [4]:
def run_command(command):
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

video_done = (VIDEO_OUT / "train_embeddings.pt").exists() and (VIDEO_OUT / "metrics.pt").exists()
if video_done and not CONFIG["force_retrain"]:
    print("Video artifacts already exist. Skipping video training.")
else:
    run_command([
        sys.executable, str(VIDEO_SCRIPT),
        "--dataset-root", str(SAMPLED_DATASET_ROOT),
        "--output-dir", str(VIDEO_OUT),
        "--epochs", str(CONFIG["video_epochs"]),
        "--batch-size", "8",
        "--lr", "1e-4",
        "--weight-decay", "1e-2",
        "--num-workers", "0",
        "--num-frames", str(CONFIG["num_frames"]),
        "--image-size", str(CONFIG["image_size"]),
        "--clip-seconds", str(CONFIG["clip_seconds"]),
        "--max-train-samples", str(CONFIG["max_train_samples"]),
        "--max-val-samples", str(CONFIG["max_val_samples"]),
        "--max-test-samples", str(CONFIG["max_test_samples"]),
    ])

Running: f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\.venv\Scripts\python.exe f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\train_mobilevit_video_deepfake.py --dataset-root f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\sampled_uniform_480_videos --output-dir f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_video --epochs 10 --batch-size 8 --lr 1e-4 --weight-decay 1e-2 --num-workers 0 --num-frames 8 --image-size 224 --clip-seconds 5 --max-train-samples 1200 --max-val-samples 160 --max-test-samples 160


In [5]:
audio_done = (AUDIO_OUT / "train_embeddings.pt").exists() and (AUDIO_OUT / "metrics.pt").exists()
if audio_done and not AUDIO_CONFIG["force_retrain"]:
    print("Audio artifacts already exist. Skipping audio training.")
else:
    run_command([
        sys.executable, str(AUDIO_SCRIPT),
        "--dataset-root", str(SAMPLED_DATASET_ROOT),
        "--output-dir", str(AUDIO_OUT),
        "--epochs", str(CONFIG["audio_epochs"]),
        "--batch-size", "8",
        "--lr", "1e-4",
        "--weight-decay", "1e-2",
        "--num-workers", "0",
        "--ffmpeg-timeout", str(AUDIO_CONFIG["ffmpeg_timeout"]),
        "--max-train-samples", str(AUDIO_CONFIG["max_train_samples"]),
        "--max-val-samples", str(AUDIO_CONFIG["max_val_samples"]),
        "--max-test-samples", str(AUDIO_CONFIG["max_test_samples"]),
    ] + (["--force-cpu"] if AUDIO_CONFIG["force_cpu"] else []))

Running: f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\.venv\Scripts\python.exe f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\train_wav2vec2_audio_deepfake.py --dataset-root f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\sampled_uniform_480_videos --output-dir f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_audio --epochs 20 --batch-size 8 --lr 1e-4 --weight-decay 1e-2 --num-workers 0 --ffmpeg-timeout 45 --max-train-samples 0 --max-val-samples 0 --max-test-samples 0


In [6]:
video_embed_train = VIDEO_OUT / "train_embeddings.pt"
audio_embed_train = AUDIO_OUT / "train_embeddings.pt"
fusion_done = (FUSION_OUT / "test_metrics.pt").exists()
if fusion_done and not FUSION_CONFIG["force_retrain"]:
    print("Fusion artifacts already exist. Skipping fusion training.")
else:
    run_command([
        sys.executable, str(FUSION_SCRIPT),
        "--video-train-embed", str(VIDEO_OUT / "train_embeddings.pt"),
        "--audio-train-embed", str(AUDIO_OUT / "train_embeddings.pt"),
        "--video-val-embed", str(VIDEO_OUT / "val_embeddings.pt"),
        "--audio-val-embed", str(AUDIO_OUT / "val_embeddings.pt"),
        "--video-test-embed", str(VIDEO_OUT / "test_embeddings.pt"),
        "--audio-test-embed", str(AUDIO_OUT / "test_embeddings.pt"),
        "--output-dir", str(FUSION_OUT),
        "--epochs", str(FUSION_CONFIG["epochs"]),
        "--batch-size", str(FUSION_CONFIG["batch_size"]),
        "--lr", str(FUSION_CONFIG["lr"]),
        "--weight-decay", str(FUSION_CONFIG["weight_decay"]),
        "--num-workers", str(FUSION_CONFIG["num_workers"]),
        "--d-model", str(FUSION_CONFIG["d_model"]),
        "--num-attn-heads", str(FUSION_CONFIG["num_attn_heads"]),
        "--dropout", str(FUSION_CONFIG["dropout"]),
        "--contrastive-weight", str(FUSION_CONFIG["contrastive_weight"]),
        "--grad-clip-norm", str(FUSION_CONFIG["grad_clip_norm"]),
        "--early-stop-patience", str(FUSION_CONFIG["early_stop_patience"]),
    ] + (["--resume"] if FUSION_CONFIG["resume"] else []))

Running: f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\.venv\Scripts\python.exe f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\train_fusion_two_head_multimodal.py --video-train-embed f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_video\train_embeddings.pt --audio-train-embed f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_audio\train_embeddings.pt --video-val-embed f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_video\val_embeddings.pt --audio-val-embed f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_audio\val_embeddings.pt --video-test-embed f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_video\test_embeddings.pt --audio-test-embed f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_audio\test_embeddings.pt --output-dir f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\loca

In [7]:
video_path = Path(INFER_CONFIG["video_path"])
checkpoint_path = Path(INFER_CONFIG["fusion_checkpoint"])
if not video_path.exists():
    candidates = sorted(SAMPLED_DATASET_ROOT.rglob("*.mp4")) + sorted(SAMPLED_DATASET_ROOT.rglob("*.MP4"))
    if candidates:
        video_path = candidates[0]
        print("Using first sampled video for inference:", video_path)

if not video_path.exists():
    print("No local video found for inference. Set INFER_CONFIG['video_path'] to a real file.")
elif not checkpoint_path.exists():
    print("Fusion checkpoint not found:", checkpoint_path)
    print("Run fusion training first or point to an existing checkpoint.")
else:
    cmd = [
        sys.executable, str(INFER_SCRIPT),
        "--video-path", str(video_path),
        "--fusion-checkpoint", str(checkpoint_path),
        "--output-json", INFER_CONFIG["output_json"],
        "--num-frames", str(INFER_CONFIG["num_frames"]),
        "--image-size", str(INFER_CONFIG["image_size"]),
        "--clip-seconds", str(INFER_CONFIG["clip_seconds"]),
        "--audio-duration-sec", str(INFER_CONFIG["audio_duration_sec"]),
        "--audio-sr", str(INFER_CONFIG["audio_sr"]),
        "--ffmpeg-timeout", str(INFER_CONFIG["ffmpeg_timeout"]),
    ]
    if INFER_CONFIG["force_cpu"]:
        cmd.append("--force-cpu")
    run_command(cmd)
    out_json = Path(INFER_CONFIG["output_json"])
    if out_json.exists():
        print(out_json.read_text())

Using first sampled video for inference: f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\sampled_uniform_480_videos\Boys\Fake_Video + Fake_Audio\Bangla\B00\FF1B00_01.mp4
Running: f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\.venv\Scripts\python.exe f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\infer_two_head_multimodal.py --video-path f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\sampled_uniform_480_videos\Boys\Fake_Video + Fake_Audio\Bangla\B00\FF1B00_01.mp4 --fusion-checkpoint f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\outputs_fusion_two_head\best_checkpoint.pt --output-json f:\Sixth Semester\DEEPfake Papers\FakeBDTeen\FakeBDTeen\local_run_outputs\single_video_prediction.json --num-frames 8 --image-size 224 --clip-seconds 5 --audio-duration-sec 10 --audio-sr 16000 --ffmpeg-timeout 45


CalledProcessError: Command '['f:\\Sixth Semester\\DEEPfake Papers\\FakeBDTeen\\.venv\\Scripts\\python.exe', 'f:\\Sixth Semester\\DEEPfake Papers\\FakeBDTeen\\FakeBDTeen\\infer_two_head_multimodal.py', '--video-path', 'f:\\Sixth Semester\\DEEPfake Papers\\FakeBDTeen\\FakeBDTeen\\local_run_outputs\\sampled_uniform_480_videos\\Boys\\Fake_Video + Fake_Audio\\Bangla\\B00\\FF1B00_01.mp4', '--fusion-checkpoint', 'f:\\Sixth Semester\\DEEPfake Papers\\FakeBDTeen\\FakeBDTeen\\local_run_outputs\\outputs_fusion_two_head\\best_checkpoint.pt', '--output-json', 'f:\\Sixth Semester\\DEEPfake Papers\\FakeBDTeen\\FakeBDTeen\\local_run_outputs\\single_video_prediction.json', '--num-frames', '8', '--image-size', '224', '--clip-seconds', '5', '--audio-duration-sec', '10', '--audio-sr', '16000', '--ffmpeg-timeout', '45']' returned non-zero exit status 1.